# Supernova driving

## Motivation

Turbulence is a key process in ISM dynamics, and has an important role on setting the star formation rate in galaxies. Several processes can generate turbulence.
In this project, we explore the nature of supernova-driven turbulence.

The goal of the project is to realise simple simulation of a piece of the ISM, modelised with very idealized assumptions.
Namely, we will start with a 200 pc uniform box of isothermal cold gas, in wich we will explode supernova at constant rate in random locations.

The energy injection will be done in the form of thermal energy.

A possible extention is to add a cooling model and magnetic field.

The result can be compared with the result of the Forced-Turbulence tutorial.

## Instalation

The tools we use are RAMSES and Osyris. More information on the codes here:
- https://github.com/nvaytet/osyris
- https://github.com/ramses-organisation/ramses

If you have followed the [general setup tutorial](https://ramses-tutorials.readthedocs.io/en/latest/Setup/general_requirements.html), osyris should already be installed. Otherwise, simply do

In [ ]:
%pip install osyris

We may also need the packages scipy (scientific functions for Python) and f90nml (a tool that help dealing with namelists, that is Fortran parameter files). Install them with 

In [ ]:
%pip install scipy f90nml

To download RAMSES, do in your terminal

In [ ]:
%%bash
git clone https://github.com/ramses-organisation/ramses.git

We will need to modify the code, so make sure to have a clean and independent copy of Ramses

To compile ramses, you need at least a fortran compiler, either fortran (GNU compiler) or ifort (Intel compiler).

## First setup : a decaying turbulence model

It is often convenient to build a setup from the one used in Ramses's test suite. In our case, to avoid starting from a completely uniform and static grid, we will use the decaying turbulence. It's a uniform isothermal box of gas, but with a initial velocity profile generating random motions. The turbulence quickly decay with time as it is not forced.

Once you downloaded RAMSES, go in the main directory (`cd ramses`)

### You can first run the test suite:

For each test, there is first a compilation (using the config.txt file of each test), then the code runs (using the namelist files .nml) and finally results are plotted using the `plot-test-name.py` routines.

There are various options for the test suite:    
```bash
#      - Run the suite in parallel (on 4 cpus):
              ./run_test_suite.sh -p 4
#      - Do not delete results data:
             ./run_test_suite.sh -d
#      - Run in verbose mode:
              ./run_test_suite.sh -v
#      - Select test number (for tests 3 to 5, and 10):
              ./run_test_suite.sh -t 3-5,10
#      - Run all tests in mhd directory:
              ./run_test_suite.sh -t mhd
#      - Run quick test suite:
              ./run_test_suite.sh -q
```
The option `-d` is quite useful if you want to work more on the data than what is proposed in the test suite. I would also recommend to use the `-y` option to use python for visualisation.

Our setup is based on the decaying turbulence test. This is the test number [5]. You will find in the directory `tests/hydro/decaying_turbulence`  the routine plot-decaying_turbulence.py that can be an alternative to the use of osyris. In particular, you can produce column density maps.

Of course, you can also play with all the other tests! Note that you can find information on the namelist parameters you can play with here: https://ramses-organisation.readthedocs.io/en/latest/wiki/Runtime_Parameters.html

In [ ]:
%%bash
cd ramses/tests
./run_test_suite.sh -t 5 -p 4

This will run the test suite for the turbulence test case with 4 MPI processes.

### You can then compile directly ramses

In the bin directory


In [ ]:
%%bash
cd ramses/bin
make clean
make NDIM=3 MPI=1 

You should have a file ramses3d that has been created.

In [ ]:
%%bash
ls ramses/bin

Copy the executable into the SN-Turbulence folder. This folder contains the file we will need for the SN driving setup. The first file you should look at is the namelist, run.nml.
For now, it is very similar to the namelist of the decaying turbulence test.

### You now need to generate the initial conditions
In the folder SN-Turbulence:

In [ ]:
%%bash
python intial_condition.py

You can then run ramses. 
If you have MPI installed you can run on 4 processors: `mpirun -np 4 ramses3d run.nml`

In [ ]:
%%bash 
./ramses3d run.nml 

You can now play with the namelist parameters. You can change the resolution (`levelmin=5`, `levelmax=5` for a 32^3 resolution simulation).
You can add more outputs using the parameter foutput in &OUTPUT_PARAMS


```config
&OUTPUT_PARAMS
noutput=2
tout=0.0,0.5
foutput=25 ! Output every 25 steps
/
```


## Use Osyris to visualise the data

You can check the documentation of osyris [here](https://osyris.readthedocs.io/en/stable/index.html).

### Plot a density slice


Modify the code below to 
- Add an extra layer with the velocity as arrows
- Change the density unit to solar masses to pc^2
- Change the colormap to inferno

The final output should look like that:

![expected_slice](slice.png)

In [ ]:
import osyris
import numpy as np
import matplotlib.pyplot as plt
import os 

path = os.path.expanduser("~/simus/SNturb_thermal/")

data = osyris.RamsesDataset(3, path=path).load() # load output number 3 
center = osyris.Vector(x=100, y=100, z=100, unit="pc") # Choose the center of the box
mu = 1.4

osyris.map(
    data["mesh"].layer("density"),  # layer 1 : density
     origin=center, direction="z", dx=200*osyris.units("pc"))

### Plot distributions

You can also directly access the data and plot basic distribution information. Here is an example for a density PDF. You can adapt this to plot the velocity PDF and a phase plot.

In [ ]:
rho = data["mesh"]["density"].to("g/cm^3").values
rhomean = np.mean(rho) # works because uniform grid
plt.figure()
plt.hist(np.log(rho/rhomean), bins=100, histtype="step", lw=2)
plt.yscale("log")
plt.xlabel("log(rho)")
plt.ylabel("DF")

## Make a movie with RAM

Ramses has a nice but 'old-fashion' module to make movies, configurable in the `MOVIE_PARAMS` block (see the documentation). The given namelist has some preconfigured option.
The output is in the folder movie1, movie2, etc. It's a collection of Fortran binary file. The code RAM (not maintained but works well) is a convienient way to read them.

You can download a working version of it by cloning the following fork:

In [ ]:
%%bash
git clone https://github.com/nbrucy/ram

We provide a config.py for RAM. Read it, adapt it to your needs and launch the movie creation with

In [ ]:
%%bash
python ram/ram3.py

You can now get the popcorn out

In [ ]:
from IPython.display import Video

Video("movie.mp4")

## Next step : adding random supernovae

We will need to modify the code to do that.


Here is a few step you can follow, in the ramses directory
- Create a new branch of ramses call `SN_driving` (optional)
- Read careffuly the new routine (`random_feedback`). This is based on the `kinetic_feedback` routine.
- Add it in the file `feedback.f90`.
- add a caller in `amr_step.f90`, next to the call to `kinetic_feedback` 
- compile and copy executable in the `SN-Turbulence` folder

```fortran

subroutine random_feedback
   use amr_commons
   use pm_commons
   use hydro_commons
   use constants, only:Myr2sec,kpc2cm,M_sun
   use mpi_mod
   use random
   implicit none
#ifndef WITHOUTMPI
   integer::info
   integer,dimension(1:ncpu)::nSN_icpu_all
   real(dp),dimension(:),allocatable::mSN_all,sSN_all,ZSN_all
   real(dp),dimension(:,:),allocatable::xSN_all,vSN_all
#endif
   !----------------------------------------------------------------------
   ! This subroutine compute the kinetic feedback due to ramdom SNII
   ! This routine is called only at coarse time step.
   ! Noé Brucy, adapted from Yohan Dubois's kinetic feedback
   !----------------------------------------------------------------------
   integer::nSN,nSN_loc,nSN_tot,iSN,ilevel,ivar
   real(dp)::scale_nH,scale_T2,scale_l,scale_d,scale_t,scale_v,t0
   real(dp)::current_time
   real(dp)::scale,dx_min,vol_min
   integer::nx_loc
   integer,dimension(:),allocatable::indSN
   real(dp),dimension(:),allocatable::mSN,sSN,ZSN,m_gas,vol_gas,ekBlast
   real(dp),dimension(:,:),allocatable::xSN,vSN,u_gas,dq

   integer ,dimension(1:ncpu,1:IRandNumSize)::allseed

   real(dp),parameter:: freqSN = 10 ! SN/Myr/kpc^2 - Exercice : change this into a namelist parameter
   real(dp),parameter:: clustering = 2
   real(dp):: freqSN_code, PoissMean
   real(dP):: xrand, yrand, zrand
 
   if(.not. hydro)return
   if(ndim.ne.3)return
 
   if(verbose)write(*,*)'Entering make_sn_random'

   ! If necessary, initialize random number generator
   if(localseed(1)==-1)then
      call rans(ncpu,iseed,allseed)
      localseed=allseed(myid,1:IRandNumSize)
   end if
   
   ! Conversion factor from user units to cgs units
   call units(scale_l,scale_t,scale_d,scale_v,scale_nH,scale_T2)
 
   ! Mesh spacing in that level
   nx_loc=(icoarse_max-icoarse_min+1)
   scale=boxlen/dble(nx_loc)
   dx_min=(0.5D0**nlevelmax)*scale
   vol_min=dx_min**ndim

   if(use_proper_time)then
      current_time=texp
   else
      current_time=t
   endif
  
   !------------------------------------------------------
   ! Root CPU computes the number of SN at this step
   !------------------------------------------------------
   if (myid==1) then

      freqSN_code = freqSN * scale_t / Myr2sec * boxlen**2 * scale_l**2 / kpc2cm**2  
      ! Poisson mean
      PoissMean=freqSN_code * dtnew(levelmin)
      call poissdev(localseed,PoissMean,nSN_tot)

      write(*,*)'-----------------------------------------------'
      write(*,*)'Number of SN to explode=',nSN_tot
      write(*,*)'-----------------------------------------------'
   end if 
 
#ifndef WITHOUTMPI
   ! Communicate the number of SN
   call MPI_BCAST(nSN_tot,1,MPI_INTEGER,0,MPI_COMM_WORLD,info)
#endif
   
   if (nSN_tot .eq. 0) return
   
   ! Allocate arrays for the position and the mass of the SN
   allocate(xSN(1:nSN_tot,1:3),vSN(1:nSN_tot,1:3))
   allocate(mSN(1:nSN_tot),sSN(1:nSN_tot),ZSN(1:nSN_tot))
   xSN=0; vSN=0; mSN=0; sSN=0; ZSN=0

 
   !------------------------------------------------------
   ! Store position and mass of the GMC into the SN array
   !------------------------------------------------------
   if(myid==1)then
      do iSN=1, nSN_tot
         call ranf(localseed, xrand)
         call ranf(localseed, yrand)
         call ranf(localseed, zrand)
         xSN(iSN,1)=xrand*boxlen
         xSN(iSN,2)=yrand*boxlen
         xSN(iSN,3)=zrand*boxlen
         vSN(iSN,1)=0.
         vSN(iSN,2)=0.
         vSN(iSN,3)=0.
         mSN(iSN)=(eta_sn+f_w)*M_sun/(scale_d*scale_l**3) ! Mass of the ejecta + mass-loaded mass
         sSN(iSN)=M_sun/(scale_d*scale_l**3) ! mass of the star
      end do
   end if
 

#ifndef WITHOUTMPI
   allocate(xSN_all(1:nSN_tot,1:3),vSN_all(1:nSN_tot,1:3),mSN_all(1:nSN_tot),sSN_all(1:nSN_tot))
   call MPI_ALLREDUCE(xSN,xSN_all,nSN_tot*3,MPI_DOUBLE_PRECISION,MPI_SUM,MPI_COMM_WORLD,info)
   call MPI_ALLREDUCE(vSN,vSN_all,nSN_tot*3,MPI_DOUBLE_PRECISION,MPI_SUM,MPI_COMM_WORLD,info)
   call MPI_ALLREDUCE(mSN,mSN_all,nSN_tot  ,MPI_DOUBLE_PRECISION,MPI_SUM,MPI_COMM_WORLD,info)
   call MPI_ALLREDUCE(sSN,sSN_all,nSN_tot  ,MPI_DOUBLE_PRECISION,MPI_SUM,MPI_COMM_WORLD,info)
   xSN=xSN_all
   vSN=vSN_all
   mSN=mSN_all
   sSN=sSN_all
   deallocate(xSN_all,vSN_all,mSN_all,sSN_all)
#endif
 
   nSN=nSN_tot
   allocate(m_gas(1:nSN),u_gas(1:nSN,1:3),vol_gas(1:nSN),dq(1:nSN,1:3),ekBlast(1:nSN))
   allocate(indSN(1:nSN))
 
   ! Compute the grid discretization effects
   call average_SN(xSN,vol_gas,dq,ekBlast,indSN,nSN)
 
   ! Modify hydro quantities to account for a Sedov blast wave
   call Sedov_blast(xSN,vSN,mSN,sSN,ZSN,indSN,vol_gas,dq,ekBlast,nSN)
 
   deallocate(xSN,vSN,mSN,sSN,ZSN,indSN,m_gas,u_gas,vol_gas,dq,ekBlast)
 
   ! Update hydro quantities for split cells
   do ilevel=nlevelmax,levelmin,-1
      call upload_fine(ilevel)
      do ivar=1,nvar
         call make_virtual_fine_dp(uold(1,ivar),ilevel)
      enddo
   enddo
 
 end subroutine random_feedback

```


## Adapt the namelist
 
 - remove the initial decaying turbulence
 - Check and uncomment the SN parameters
 - Check the cooling parameters : make the simulation non isothermal 
 - [advanced] make it possible to change the SN frequency as a parameter


## Further analysis

- Use osyris to compute the global velocity dispersion (Mach number)
- 2D power spectrum on slices
- 3D power spectrum on cubes

## Improved simulations

- [easy] Use `cooling_ism` for a more realstic ISM. Check the phase plot. Beware overcooling!
- [easy] Improve the resolution. Use AMR and decide on a refinement criterion
- [intermediate] Account for clustered feedback
- [intermediate] Run with MHD
- [more challenging] Add gravity, and explode on density peaks